In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "4,5,6,7"  # Set visible GPUs
os.environ["TOKENIZERS_PARALLELISM"] = "true"

In [ ]:
# Define a custom CtrlG processor for Qwen3-8B-Base model

import ctrlg_custom
from vllm.config import VllmConfig
import torch
import os

SAMPLING_DATASET_PATH = "openvlthinker_medium_boxed.json"
# Model path or model name from Hugging Face
# BASE_MODEL_PATH = "ydeng9/OpenVLThinker-7B-v1.2"
BASE_MODEL_PATH = "Qwen/Qwen2.5-VL-7B-Instruct"

# Settings
# base_model_path = "billkunghappy/Qwen3-8B-Base-Dapo-V6-S60"
# hmm_model_path = "billkunghappy/hmm_Qwen3-8B-Base-Dapo-V6-S60-MERGED-4096-Step1000"
# ctrlg_dict_name = "CONSTRAINTS_DICT_V4"
# min_new_tokens = 1
# max_new_tokens = 1024*8
# alpha = 2.0

hmm_model_path = "/data2/ponienkung/Ctrl-G/distillation_vllm/workspace/models/hmm_Qwen2.5-VL-7B-Instruct_openvlthinker_medium_boxed_4096/checkpoint-2000"
ctrlg_dict_name = "CONSTRAINTS_DICT_V4"
min_new_tokens = 1
max_new_tokens = 1024*2
alpha = 2.0
soft_constraints = True

# Calculated
base_model_name = os.path.basename(BASE_MODEL_PATH)
hmm_name = os.path.basename(hmm_model_path)

output_path = f"outputs/{base_model_name}_{hmm_name}_{ctrlg_dict_name}_A{alpha}_Soft-{soft_constraints}.json"
constraints_dict = getattr(ctrlg_custom, ctrlg_dict_name)


# Define the custom CtrlG processor class
class Qwen3BaseCtrlgProcessorEval(ctrlg_custom.CtrlgBatchLogitsProcessor):
    """Qwen3-8B-Base specific configuration"""
    
    def __init__(self, vllm_config: VllmConfig, device: torch.device, is_pin_memory: bool):
        super().__init__(
            vllm_config=vllm_config,
            device=device,
            is_pin_memory=is_pin_memory,
            hmm_model_path=hmm_model_path,
            tokenizer_path=BASE_MODEL_PATH,
            min_new_tokens=min_new_tokens,
            max_new_tokens=max_new_tokens,
            alpha=alpha,
            constraints_dict=constraints_dict,
            soft_constraints=soft_constraints
        )
        print("⚙️ Initialized Qwen3BaseCtrlgProcessorEval")

In [ ]:
# Initialize VLLM LLM with custom CtrlG processor
from vllm import LLM, SamplingParams
import os
import torch

available_gpus = torch.cuda.device_count()

# Initialize the LLM with the custom logits processor
llm = LLM(
    model=BASE_MODEL_PATH,  # Base model path
    tensor_parallel_size=available_gpus,  # Use all available GPUs
    trust_remote_code=True,
    dtype="auto",  # Let VLLM decide the optimal dtype
    max_model_len=2048,  # Adjust based on your requirements
    gpu_memory_utilization=0.6,
    mm_processor_kwargs={"max_pixels": 262144, "min_pixels": 262144},
    # Register our custom logits processor
    logits_processors=[Qwen3BaseCtrlgProcessorEval]
)

print("🚀 VLLM LLM initialized with custom CtrlG processor!")


In [ ]:
# Define sampling parameters, use the `rollout` parameters
sampling_params = SamplingParams(
    temperature=1.0,
    top_p=0.9,
    top_k=-1,
    max_tokens=1024*2,
    min_tokens=1,
    n=1, # cannot set to other value
)


In [ ]:
# For multi-modal data

import base64
import io
from PIL import Image

def decode_base64_image(img_b64: str, mime_type: str = None) -> Image.Image:
    """
    Decode a base64-encoded image and return a PIL Image.
    
    Args:
        img_b64 (str): Base64 string of the image. 
            Can be either a raw base64 string or a data URL starting with 'data:'.
        mime_type (str): Optional MIME type (e.g. 'image/png'). 
            Only needed if img_b64 doesn't include the header.

    Returns:
        PIL.Image.Image: The decoded image.
    """
    # If it includes a data URL header, remove it
    if img_b64.startswith("data:"):
        header, img_b64 = img_b64.split(",", 1)
        # Extract MIME type if not explicitly provided
        if not mime_type and ";" in header:
            mime_type = header.split(";")[0].split(":")[1]

    # Decode the base64 string to bytes
    img_bytes = base64.b64decode(img_b64)

    # Wrap bytes in a BytesIO and open via PIL
    image = Image.open(io.BytesIO(img_bytes))

    return image

# Example usage
# Case 1: If you already have a MIME type
# image = decode_base64_image(img_b64, mime_type="image/png")

# Case 2: If you have a full data URL
# image = decode_base64_image("data:image/jpeg;base64,/9j/4AAQSkZJRgABAQAAAQABAAD...")
# image.show()


In [ ]:
# Load the dataset
import json
from copy import deepcopy
import random
from transformers import AutoProcessor, AutoTokenizer

# dataset_path = "sampled_20_dapo_prompts.json"
dataset_path = "openvlthinker_shuffled.json"
processor = AutoProcessor.from_pretrained(BASE_MODEL_PATH)
# tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
# Number of repeated data
repeat_n = 1

data_num = 5
test_data = json.load(open(dataset_path, 'r'))

# Random sample a number of data to use
test_data = random.sample(test_data, min(len(test_data), data_num))

if type(test_data[0]) == dict:
    prompts = []
    for t in test_data:
        msg = [{
            "role": "user",
            "content": [
                {"type": "image", "image": "image-1"},
                {"type": "text", "text": t['prompt']}
            ]
        }]
        prompt_text = processor.apply_chat_template(
            msg, add_generation_prompt=True, tokenize=False
        )
        # print(prompt_text)    
        image = decode_base64_image(t['img_b64'], mime_type=t['mime_type'])
        prompt = {
            "prompt": prompt_text,
            "multi_modal_data": {"image": [image]},
        }
        prompts.append(prompt)

elif type(test_data) == str:
    prompts = [processor.apply_chat_template(t, add_generation_prompt=True, tokenize=False) for t in test_data]
else:
    raise ValueError("Unsupported data format")

In [ ]:
# Organize prompts and sampling params

def repeat_interleave(lst, n):
    return [item for item in lst for _ in range(n)]

prompts_n = repeat_interleave(prompts, repeat_n)

all_prompts = []
all_sampling_params = []
for cnst_type in constraints_dict.keys():
    all_prompts.extend(prompts_n)
    sp = deepcopy(sampling_params)
    setattr(sp, 'extra_args', {"clp_id": cnst_type})
    all_sampling_params.extend([sp]*len(prompts_n))

outputs = llm.generate(all_prompts, all_sampling_params)


In [ ]:
# Collect all responses
responses = []
for output in outputs:
    for sample_id in range(len(output.outputs)):
        responses.append(output.outputs[sample_id].text) # For vllm-v0, change tuple to list

print(f"Generated {len(responses)} responses for constraint type: {cnst_type}")

results = [{"output": res, "cnst_type": sp.extra_args["clp_id"]} for res, sp in zip(responses, all_sampling_params)]

# Write responses to a file
json.dump(results, open(output_path, 'w'), indent=4)
print(f"Responses saved to {output_path}")

## Plot comparison results

In [ ]:
import json
import re
from collections import Counter

def extract_answers(output_text):
    """
    Extract all answers after 'Answer:'.
    """
    answers = re.findall(r"Answer:\s*([^\n\r]*)", output_text)
    return [a.strip() for a in answers if a.strip()]

# def has_repeated_same_answer(output_text, threshold):
#     answers = extract_answers(output_text)
#     if not answers:
#         return False
#     counts = Counter(answers)
#     return any(c > threshold for c in counts.values())

def has_consecutive_same_answer(output_text, threshold):
    """
    Stricter: check if the SAME answer appears consecutively.
    """
    answers = extract_answers(output_text)
    num_consecutive = 0
    for i in range(1, len(answers)):
        if answers[i] == answers[i-1]:  # consecutive match
            num_consecutive += 1
            if num_consecutive > threshold:
                return True
    return False

In [ ]:
# Load all json files in "outputs/", each file is a list of dict with keys: "output" and "cnst_type".
# For each of the files, compute the percentage of outputs that have consecutive answers for each constraint type.
# Plot the results in a bar chart. Each bar group represents a constraint type, and within each group, there are n bars, which n is the number of files.
# Make it a function which takes thres as a parameter.

import matplotlib.pyplot as plt
import numpy as np
import glob
from collections import defaultdict

def plot_consecutive_answers_comparison(threshold=1):
    """
    Plot comparison of consecutive answer percentages across different output files.
    
    Args:
        threshold (int): Minimum number of consecutive same answers to count as repetition
    """
    # Find all JSON files in outputs directory
    json_files = glob.glob("outputs/*.json")
    
    if not json_files:
        print("No JSON files found in outputs/ directory")
        return
    
    print(f"Found {len(json_files)} JSON files: {[os.path.basename(f) for f in json_files]}")
    
    # Dictionary to store results: {constraint_type: {file_name: percentage}}
    results = defaultdict(dict)
    
    # Process each file
    for json_file in json_files:
        file_name = os.path.basename(json_file).replace('.json', '')
        
        try:
            with open(json_file, 'r') as f:
                data = json.load(f)
            
            # Group by constraint type
            constraint_groups = defaultdict(list)
            for item in data:
                constraint_groups[item['cnst_type']].append(item['output'])
            
            # Calculate percentage for each constraint type
            for cnst_type, outputs in constraint_groups.items():
                consecutive_count = sum(1 for output in outputs if has_consecutive_same_answer(output, threshold))
                percentage = (consecutive_count / len(outputs)) * 100 if outputs else 0
                results[cnst_type][file_name] = percentage
                
        except Exception as e:
            print(f"Error processing {json_file}: {e}")
            continue
    
    if not results:
        print("No valid data found to plot")
        return
    
    # Prepare data for plotting
    constraint_types = list(results.keys())
    file_names = list(set(file_name for cnst_data in results.values() for file_name in cnst_data.keys()))
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(15, 8))
    
    # Set up bar positions
    x = np.arange(len(constraint_types))
    width = 0.8 / len(file_names)  # Width of bars
    
    # Plot bars for each file
    for i, file_name in enumerate(file_names):
        percentages = []
        for cnst_type in constraint_types:
            percentages.append(results[cnst_type].get(file_name, 0))
        
        bars = ax.bar(x + i * width, percentages, width, 
                     label=file_name, alpha=0.8)
        
        # Add value labels on bars
        for bar, percentage in zip(bars, percentages):
            if percentage > 0:
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                       f'{percentage:.1f}%', ha='center', va='bottom', fontsize=8)
    
    # Customize the plot
    ax.set_xlabel('Constraint Types', fontsize=12)
    ax.set_ylabel('Percentage of Outputs with Consecutive Answers (%)', fontsize=12)
    ax.set_title(f'Consecutive Answer Repetition Comparison (Threshold: {threshold})', fontsize=14)
    ax.set_xticks(x + width * (len(file_names) - 1) / 2)
    ax.set_xticklabels(constraint_types, rotation=45, ha='right')
    ax.legend(bbox_to_anchor=(0.5, -0.15), loc='upper center', ncol=min(len(file_names), 3))
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 100)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print(f"\nSummary (Threshold: {threshold}):")
    for cnst_type in constraint_types:
        print(f"\n{cnst_type}:")
        for file_name, percentage in results[cnst_type].items():
            print(f"  {file_name}: {percentage:.1f}%")

# Example usage
plot_consecutive_answers_comparison(threshold=3)